# FastSpeech 2 — English (Svarah) & Telugu (IndicTTS) on Google Colab

1. Runtime → Change runtime type → **GPU** (T4 is suitable).
2. Upload `FastSpeech2_project.zip` to the root of your Google Drive (`MyDrive`).
3. Set `LANG` and `ENGLISH_VERSION` below, then run all cells. Run the notebook separately for English and Telugu.

Project files, checkpoints and results are stored on Google Drive. Raw and preprocessed audio are stored on temporary Colab storage, so they may need to be downloaded and preprocessed again after a runtime reset. Training resumes from the selected checkpoint directory's `latest.pth` when available.

In [ ]:
LANG = 'English'       # 'English' or 'Telugu'
ENGLISH_VERSION = 'v3' # 'original' or 'v3' (only used for English)
MAX_STEPS = None       # None uses the selected config's total_step
HF_TOKEN = ''          # Required for English (Svarah is gated)

In [ ]:
import os
import sys
import subprocess
from google.colab import drive

if LANG not in ('English', 'Telugu'):
    raise ValueError("LANG must be 'English' or 'Telugu'")
if LANG == 'English' and ENGLISH_VERSION not in ('original', 'v3'):
    raise ValueError("ENGLISH_VERSION must be 'original' or 'v3'")

drive.mount('/content/drive')
!nvidia-smi

In [ ]:
ROOT = '/content/drive/MyDrive/FastSpeech2_project'
ZIP_PATH = '/content/drive/MyDrive/FastSpeech2_project.zip'

if not os.path.isdir(ROOT):
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(f'Upload the project ZIP to MyDrive: {ZIP_PATH}')
    subprocess.run(['unzip', '-q', ZIP_PATH, '-d', '/content/drive/MyDrive/'], check=True)

if not os.path.isdir(ROOT):
    raise FileNotFoundError(f'Project folder not found at {ROOT}. Check the ZIP structure.')

os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

### Select the experiment

English original uses `config.yaml` and `checkpoints/`. English v3 uses `config_v3.yaml` and `checkpoints_v3/`. Telugu uses its own `config.yaml` and `checkpoints/`.

English v2 is not included because its decoder architecture differs from the default English inference configuration.

In [ ]:
if LANG == 'Telugu':
    CONFIG = 'config.yaml'
    CKPT_DIR = 'checkpoints'
    RESULT_DIR = '../Results/Telugu'
    DEFAULT_STEPS = 50000
elif ENGLISH_VERSION == 'v3':
    CONFIG = 'config_v3.yaml'
    CKPT_DIR = 'checkpoints_v3'
    RESULT_DIR = '../Results/English_v3'
    DEFAULT_STEPS = 90000
else:
    CONFIG = 'config.yaml'
    CKPT_DIR = 'checkpoints'
    RESULT_DIR = '../Results/English'
    DEFAULT_STEPS = 50000

STEPS = DEFAULT_STEPS if MAX_STEPS is None else int(MAX_STEPS)
LANG_DIR = os.path.join(ROOT, LANG)

print('Language:', LANG)
print('Configuration:', CONFIG)
print('Checkpoint directory:', CKPT_DIR)
print('Results directory:', RESULT_DIR)
print('Training steps:', STEPS)

Raw and preprocessed audio are kept on the local Colab disk for faster processing. The project `data` directory is linked to `/content/data`. Existing data is preserved as `data_old` rather than deleted.

In [ ]:
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'
LOCAL_DATA = '/content/data'
PROJECT_DATA = os.path.join(ROOT, 'data')
os.makedirs(LOCAL_DATA, exist_ok=True)

if os.path.islink(PROJECT_DATA):
    if os.path.realpath(PROJECT_DATA) != os.path.realpath(LOCAL_DATA):
        os.unlink(PROJECT_DATA)
elif os.path.exists(PROJECT_DATA):
    BACKUP_DATA = PROJECT_DATA + '_old'
    if os.path.exists(BACKUP_DATA):
        raise FileExistsError(f'{BACKUP_DATA} already exists. Inspect it before continuing.')
    os.rename(PROJECT_DATA, BACKUP_DATA)

if not os.path.lexists(PROJECT_DATA):
    os.symlink(LOCAL_DATA, PROJECT_DATA)

os.chdir(LANG_DIR)
print('Working directory:', os.getcwd())

### 1. Download and prepare the dataset

For English, accept the Svarah dataset terms at `huggingface.co/datasets/ai4bharat/Svarah` and provide a Hugging Face access token. Telugu preparation does not require this token.

In [ ]:
raw_train = f'../data/{LANG}/raw/train.tsv'

if not os.path.exists(raw_train):
    if LANG == 'English':
        if not HF_TOKEN:
            raise ValueError('Set HF_TOKEN before preparing the gated Svarah dataset.')
        subprocess.run([sys.executable, 'prepare_data.py', '--hf_token', HF_TOKEN], check=True)
    else:
        subprocess.run([sys.executable, 'prepare_data.py'], check=True)

stats_file = f'../data/{LANG}/raw/selection_stats.json'
if os.path.isfile(stats_file):
    with open(stats_file) as f:
        print(f.read())
else:
    print('Selection statistics not found:', stats_file)

In [ ]:
preprocessed_train = f'../data/{LANG}/preprocessed/train.txt'

if not os.path.exists(preprocessed_train):
    subprocess.run([sys.executable, 'preprocess.py', '--workers', '2'], check=True)

summary_file = f'../data/{LANG}/preprocessed/summary.json'
if os.path.isfile(summary_file):
    with open(summary_file) as f:
        print(f.read())
else:
    print('Preprocessing summary not found:', summary_file)

### 2. Train

The selected configuration determines the checkpoint directory. `MAX_STEPS = None` uses the total training steps specified in the configuration. Training resumes from the latest checkpoint when the training code finds one.

In [ ]:
train_cmd = [sys.executable, 'train.py', '--config', CONFIG, '--workers', '2']
if MAX_STEPS is not None:
    train_cmd.extend(['--max_steps', str(STEPS)])

subprocess.run(train_cmd, check=True)

### 3. Run inference and listen to generated audio

The supplied inference scripts load `config.yaml` by default. This cell explicitly selects the trained checkpoint and matching results directory. English v3 has the same model architecture as the original English configuration.

In [ ]:
import glob
from IPython.display import Audio, display

checkpoint = os.path.join(CKPT_DIR, 'final_fp16.pth')
if not os.path.isfile(checkpoint):
    raise FileNotFoundError(f'Inference checkpoint not found: {checkpoint}')

subprocess.run([
    sys.executable, 'inference.py',
    '--ckpt', checkpoint,
    '--out', RESULT_DIR
], check=True)

audio_files = sorted(glob.glob(f'{RESULT_DIR}/test_generated_audios/*.wav'))
if not audio_files:
    print('No generated WAV files found. Check the inference output above.')
else:
    for wav_path in audio_files:
        print(wav_path)
        display(Audio(filename=wav_path))

### 4. View training and validation loss curves

In [ ]:
from IPython.display import Image, display

for filename in ('training_loss.png', 'validation_loss.png'):
    plot_path = os.path.join(RESULT_DIR, filename)
    if os.path.isfile(plot_path):
        display(Image(filename=plot_path))
    else:
        print('Plot not found:', plot_path)